# Chapter 13 Computational Lab
## Moment Generating and Characteristic Functions

This notebook accompanies Chapter 13 of *Probability Theory with Python and AI*.

The central operational principle of the chapter is:

$$
\boxed{
\text{independent sums}
\quad\longleftrightarrow\quad
\text{products of transforms}.
}
$$

Probability generating functions from Chapter 8, moment generating functions, and characteristic functions are different versions of the same encoding idea.

### Learning goals

By the end of the lab you should be able to:

1. connect PGFs, MGFs and characteristic functions;
2. state precisely what it means for an MGF to exist near zero;
3. use MGF derivatives to recover moments;
4. understand why MGF existence near zero implies all absolute moments are finite;
5. derive MGFs of Bernoulli, binomial, Poisson, exponential, gamma and normal laws;
6. use independence to turn sums into products of MGFs;
7. apply Chernoff bounds using an MGF;
8. explain why the standard Cauchy distribution has no MGF near zero;
9. define characteristic functions as bounded complex expectations;
10. use the basic CF identities, including conjugacy, affine transformation and uniform continuity;
11. recover existing moments from CF derivatives;
12. compute CFs of the principal distributions;
13. understand the Gaussian ODE proof of the normal CF;
14. understand the real-analysis proof that the standard Cauchy CF is $e^{-|t|}$;
15. use characteristic functions for independent sums without any MGF-existence assumption;
16. explain why uniqueness is essential for distributional identification;
17. understand the Gaussian-smoothing idea behind CF uniqueness;
18. prove Cauchy stability under averaging;
19. understand why MGF equality identifies distributions only when both MGFs exist near zero;
20. compare empirical MGFs and empirical characteristic functions numerically;
21. audit AI-generated transform arguments.

> **Forward link.** Characteristic functions will become the main analytic engine in the proof of the central limit theorem in Chapter 14.


## 0. Setup

The formulas are implemented directly rather than through black-box transform routines.


In [ ]:
from fractions import Fraction
from math import comb
import cmath
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def empirical_mgf(sample, t):
    sample = np.asarray(sample, dtype=float)
    return np.mean(np.exp(t*sample))


def empirical_cf(sample, t):
    sample = np.asarray(sample, dtype=float)
    return np.mean(np.exp(1j*t*sample))


def bernoulli_mgf(t, p):
    return 1-p+p*np.exp(t)


def bernoulli_cf(t, p):
    return 1-p+p*np.exp(1j*t)


def binomial_mgf(t, n, p):
    return (1-p+p*np.exp(t))**n


def binomial_cf(t, n, p):
    return (1-p+p*np.exp(1j*t))**n


def poisson_mgf(t, lam):
    return np.exp(lam*(np.exp(t)-1))


def poisson_cf(t, lam):
    return np.exp(lam*(np.exp(1j*t)-1))


def exponential_mgf(t, lam):
    t = np.asarray(t, dtype=float)
    return np.where(t < lam, lam/(lam-t), np.inf)


def exponential_cf(t, lam):
    t = np.asarray(t, dtype=float)
    return lam/(lam-1j*t)


def gamma_mgf(t, alpha, lam):
    t = np.asarray(t, dtype=float)
    return np.where(
        t < lam,
        (lam/(lam-t))**alpha,
        np.inf,
    )


def normal_mgf(t, mu, sigma):
    t = np.asarray(t, dtype=float)
    return np.exp(mu*t + 0.5*sigma*sigma*t*t)


def normal_cf(t, mu, sigma):
    t = np.asarray(t, dtype=float)
    return np.exp(1j*mu*t - 0.5*sigma*sigma*t*t)


def cauchy_cf(t):
    t = np.asarray(t, dtype=float)
    return np.exp(-np.abs(t))


def uniform_minus1_1_cf(t):
    t = np.asarray(t, dtype=float)
    out = np.ones_like(t)
    mask = np.abs(t) > 1e-14
    out[mask] = np.sin(t[mask])/t[mask]
    return out


def gaussian_density(x, epsilon):
    x = np.asarray(x, dtype=float)
    return np.exp(-x*x/(2*epsilon*epsilon))/(math.sqrt(2*math.pi)*epsilon)


def normal_cdf_scalar(x):
    return 0.5*(1+math.erf(x/math.sqrt(2)))


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Transform tools are ready."
    "</div>"
))


## 1. One idea, three transforms

For a law concentrated on $\mathbb N_0$, Chapter 8 defined the probability generating function

$$
G_X(s)
=
\sum_{k=0}^{\infty}
P(X=k)s^k.
$$

The same power series converges absolutely for every complex $s$ with $|s|\le1$.

Therefore, for every real $t$,

$$
\boxed{
\varphi_X(t)
=
G_X(e^{it}).
}
$$

Whenever the MGF is finite at $t$,

$$
\boxed{
M_X(t)
=
G_X(e^t).
}
$$


### Bernoulli bridge

For $X\sim\operatorname{Bernoulli}(p)$,

$$
G_X(s)=1-p+ps,
$$

so

$$
G_X(e^t)
=
1-p+pe^t
=
M_X(t),
$$

and

$$
G_X(e^{it})
=
1-p+pe^{it}
=
\varphi_X(t).
$$


In [ ]:
bridge_p = widgets.FloatSlider(value=0.3, min=0, max=1, step=0.01, description="p")
bridge_t = widgets.FloatSlider(value=0.8, min=-3, max=3, step=0.1, description="t")
bridge_output = widgets.Output()


def update_bridge(*_):
    with bridge_output:
        clear_output(wait=True)

        p = bridge_p.value
        t = bridge_t.value

        G_et = 1-p+p*np.exp(t)
        G_eit = 1-p+p*np.exp(1j*t)

        display(Math(r"G_X(e^t)=" + f"{G_et:.8f}"))
        display(Markdown(
            f"$G_X(e^{{it}})$ = **{G_eit.real:.6f} {G_eit.imag:+.6f}i**"
        ))


for control in (bridge_p, bridge_t):
    control.observe(update_bridge, names="value")

display(widgets.VBox([
    widgets.HBox([bridge_p, bridge_t]),
    bridge_output,
]))
update_bridge()


## 2. Moment generating functions

At every real $t$ for which the expectation is finite,

$$
\boxed{
M_X(t)
=
\mathbb E[e^{tX}].
}
$$

Every random variable satisfies

$$
M_X(0)=1.
$$

Therefore existence only at $t=0$ is not informative.


### Existence near zero

The useful condition is:

$$
\boxed{
\exists\,\delta>0
\quad\text{such that}\quad
M_X(t)<\infty
\text{ for every }|t|<\delta.
}
$$

This two-sided neighborhood is what supports the moment theory.


### Affine transformation

For constants $a,b$,

$$
\boxed{
M_{aX+b}(t)
=
e^{bt}M_X(at),
}
$$

whenever the expressions are finite.


In [ ]:
affine_a = widgets.FloatSlider(value=-2, min=-4, max=4, step=0.25, description="a")
affine_b = widgets.FloatSlider(value=5, min=-10, max=10, step=0.5, description="b")
affine_t = widgets.FloatSlider(value=0.4, min=-1, max=1, step=0.05, description="t")
affine_output = widgets.Output()


def update_affine_mgf(*_):
    with affine_output:
        clear_output(wait=True)

        # X ~ N(2,9)
        a = affine_a.value
        b = affine_b.value
        t = affine_t.value

        direct_mu = a*2+b
        direct_sigma = abs(a)*3

        lhs = normal_mgf(t, direct_mu, direct_sigma)
        rhs = math.exp(b*t)*normal_mgf(a*t, 2, 3)

        display(Math(r"M_{aX+b}(t)=" + f"{float(lhs):.8f}"))
        display(Math(r"e^{bt}M_X(at)=" + f"{float(rhs):.8f}"))


for control in (affine_a, affine_b, affine_t):
    control.observe(update_affine_mgf, names="value")

display(widgets.VBox([
    widgets.HBox([affine_a, affine_b, affine_t]),
    affine_output,
]))
update_affine_mgf()


## 3. Why the MGF generates moments

If $M_X(t)$ is finite on an open interval around zero, then $M_X$ is infinitely differentiable there and

$$
\boxed{
M_X^{(n)}(t)
=
\mathbb E[X^n e^{tX}].
}
$$

In particular,

$$
\boxed{
M_X^{(n)}(0)
=
\mathbb E[X^n].
}
$$

The point is not formal differentiation: MGF finiteness near zero provides an integrable dominating random variable.


### Mean and variance

The first two derivatives give

$$
\boxed{
\mathbb E[X]
=
M_X'(0),
}
$$

and

$$
\boxed{
\operatorname{Var}(X)
=
M_X''(0)
-
\bigl(M_X'(0)\bigr)^2.
}
$$


In [ ]:
mgf_p = widgets.FloatSlider(value=0.35, min=0, max=1, step=0.01, description="p")
mgf_moment_output = widgets.Output()


def update_bernoulli_moments(*_):
    with mgf_moment_output:
        clear_output(wait=True)

        p = mgf_p.value

        # M(t)=1-p+p e^t; first and second derivative at zero are p.
        mean = p
        second = p
        var = second-mean**2

        display(Math(r"M_X'(0)=" + f"{mean:.6f}"))
        display(Math(r"M_X''(0)=" + f"{second:.6f}"))
        display(Math(r"\operatorname{Var}(X)=" + f"{var:.6f}"))


mgf_p.observe(update_bernoulli_moments, names="value")
display(widgets.VBox([mgf_p, mgf_moment_output]))
update_bernoulli_moments()


### MGF existence near zero is strong

If an MGF is finite on a neighborhood of zero, then for every integer $n\ge1$,

$$
\boxed{
\mathbb E[|X|^n]
<
\infty.
}
$$

A short domination argument uses

$$
|x|^n
\le
C_n e^{r|x|}
$$

and

$$
e^{r|X|}
\le
e^{rX}+e^{-rX}.
$$


## 4. MGFs of important distributions

| Distribution | $M_X(t)$ | Domain displayed in the chapter |
|---|---|---|
| Bernoulli$(p)$ | $1-p+pe^t$ | all $t\in\mathbb R$ |
| Binomial$(n,p)$ | $(1-p+pe^t)^n$ | all $t\in\mathbb R$ |
| Poisson$(\lambda)$ | $\exp(\lambda(e^t-1))$ | all $t\in\mathbb R$ |
| Exponential$(\lambda)$ | $\lambda/(\lambda-t)$ | $t<\lambda$ |
| Gamma$(\alpha,\lambda)$ | $(\lambda/(\lambda-t))^\alpha$ | $t<\lambda$ |
| Normal$(\mu,\sigma^2)$ | $\exp(\mu t+\sigma^2t^2/2)$ | all $t\in\mathbb R$ |

For the gamma law, $\lambda$ is the **rate**, as in Chapter 10.


### Domains matter

For

$$
X\sim\operatorname{Exp}(\lambda),
$$

$$
M_X(t)
=
\frac{\lambda}{\lambda-t}
$$

only for

$$
t<\lambda.
$$

At $t=\lambda$ the defining integral already diverges.


In [ ]:
domain_lam = widgets.FloatSlider(value=2.0, min=0.5, max=5, step=0.1, description="lambda")
domain_output = widgets.Output()


def update_domain(*_):
    with domain_output:
        clear_output(wait=True)

        lam = domain_lam.value
        t = np.linspace(-3, lam-0.03, 900)
        M = lam/(lam-t)

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.plot(t,M)
        ax.axvline(lam,linestyle="--")
        ax.set_xlabel("t")
        ax.set_ylabel("M_X(t)")
        ax.set_title("Exponential MGF and its right-hand domain boundary")
        ax.set_ylim(0, min(20, np.quantile(M,0.98)))
        plt.show()

        display(Math(r"M_X(t)<\infty\quad\Longleftrightarrow\quad t<\lambda"))


domain_lam.observe(update_domain, names="value")
display(widgets.VBox([domain_lam,domain_output]))
update_domain()


### Gamma moments from the transform

For

$$
X\sim\operatorname{Gamma}(\alpha,\lambda),
$$

$$
M_X(t)
=
\left(
\frac{\lambda}{\lambda-t}
\right)^\alpha.
$$

Differentiation at zero gives

$$
\boxed{
\mathbb E[X]
=
\frac{\alpha}{\lambda},
}
$$

and

$$
\boxed{
\operatorname{Var}(X)
=
\frac{\alpha}{\lambda^2}.
}
$$


In [ ]:
gamma_alpha = widgets.FloatSlider(value=3, min=0.5, max=10, step=0.5, description="alpha")
gamma_lambda = widgets.FloatSlider(value=2, min=0.5, max=5, step=0.25, description="lambda")
gamma_output = widgets.Output()


def update_gamma_transform(*_):
    with gamma_output:
        clear_output(wait=True)

        alpha = gamma_alpha.value
        lam = gamma_lambda.value

        mean = alpha/lam
        variance = alpha/(lam*lam)

        display(Math(r"\mathbb E[X]=" + f"{mean:.6f}"))
        display(Math(r"\operatorname{Var}(X)=" + f"{variance:.6f}"))


for control in (gamma_alpha,gamma_lambda):
    control.observe(update_gamma_transform, names="value")

display(widgets.VBox([
    widgets.HBox([gamma_alpha,gamma_lambda]),
    gamma_output,
]))
update_gamma_transform()


### Normal MGF

For

$$
X\sim N(\mu,\sigma^2),
$$

completion of the square yields

$$
\boxed{
M_X(t)
=
\exp\left(
\mu t+\frac12\sigma^2t^2
\right).
}
$$

The first two derivatives encode $\mu$ and $\sigma^2$.


## 5. Independent sums: products replace convolution

If $X$ and $Y$ are independent, then

$$
\boxed{
M_{X+Y}(t)
=
M_X(t)M_Y(t),
}
$$

at every $t$ where the relevant expectations are finite.

For independent $X_1,\ldots,X_n$,

$$
\boxed{
M_{X_1+\cdots+X_n}(t)
=
\prod_{j=1}^nM_{X_j}(t).
}
$$

Independence is used exactly when the expectation of the product is factorized.


### Transform diagram

The same operation can be viewed in two spaces:

$$
\boxed{
\text{convolution of probability laws}
\quad\longleftrightarrow\quad
\text{multiplication of transforms}.
}
$$

Repeated convolution may be cumbersome. Repeated multiplication is usually simple.


### Poisson parameters add

If

$$
X\sim\operatorname{Poisson}(\lambda_1),
\qquad
Y\sim\operatorname{Poisson}(\lambda_2)
$$

independently, then

$$
M_{X+Y}(t)
=
\exp\left(
(\lambda_1+\lambda_2)(e^t-1)
\right).
$$

Once MGF uniqueness is available,

$$
\boxed{
X+Y
\sim
\operatorname{Poisson}(\lambda_1+\lambda_2).
}
$$


In [ ]:
sum_lam1 = widgets.FloatSlider(value=2, min=0.2, max=6, step=0.2, description="lambda1")
sum_lam2 = widgets.FloatSlider(value=3, min=0.2, max=6, step=0.2, description="lambda2")
sum_t = widgets.FloatSlider(value=0.4, min=-1, max=1, step=0.05, description="t")
sum_output = widgets.Output()


def update_poisson_sum(*_):
    with sum_output:
        clear_output(wait=True)

        l1 = sum_lam1.value
        l2 = sum_lam2.value
        t = sum_t.value

        product_value = poisson_mgf(t,l1)*poisson_mgf(t,l2)
        combined = poisson_mgf(t,l1+l2)

        display(Math(r"M_X(t)M_Y(t)=" + f"{float(product_value):.8f}"))
        display(Math(r"M_{\mathrm{Pois}(\lambda_1+\lambda_2)}(t)=" + f"{float(combined):.8f}"))


for control in (sum_lam1,sum_lam2,sum_t):
    control.observe(update_poisson_sum, names="value")

display(widgets.VBox([
    widgets.HBox([sum_lam1,sum_lam2,sum_t]),
    sum_output,
]))
update_poisson_sum()


### Gamma shapes add at a common rate

If

$$
X\sim\operatorname{Gamma}(\alpha,\lambda),
\qquad
Y\sim\operatorname{Gamma}(\beta,\lambda),
$$

independently, then

$$
\boxed{
X+Y
\sim
\operatorname{Gamma}(\alpha+\beta,\lambda).
}
$$

The common-rate assumption is what makes the two factors combine into a single gamma MGF.


### Normal means and variances add

If independent

$$
X\sim N(\mu_1,\sigma_1^2),
\qquad
Y\sim N(\mu_2,\sigma_2^2),
$$

then

$$
\boxed{
X+Y
\sim
N(
\mu_1+\mu_2,
\sigma_1^2+\sigma_2^2
).
}
$$


## 6. MGFs and Chernoff bounds

If $t>0$ and $M_X(t)<\infty$, then Markov's inequality applied to $e^{tX}$ gives

$$
\boxed{
P(X\ge a)
\le
e^{-ta}M_X(t).
}
$$

Optimizing over admissible $t>0$ gives

$$
\boxed{
P(X\ge a)
\le
\inf_{t>0}
e^{-ta}M_X(t).
}
$$


### Gaussian Chernoff bound

For $Z\sim N(0,1)$,

$$
M_Z(t)=e^{t^2/2}.
$$

Thus

$$
P(Z\ge a)
\le
\exp\left(
-ta+\frac{t^2}{2}
\right).
$$

The exponent is minimized at $t=a$, giving

$$
\boxed{
P(Z\ge a)
\le
e^{-a^2/2}.
}
$$

This is a bound, not the exact normal-tail probability.


In [ ]:
chernoff_a = widgets.FloatSlider(value=2, min=0.2, max=5, step=0.1, description="a")
chernoff_t = widgets.FloatSlider(value=2, min=0.05, max=6, step=0.05, description="t")
chernoff_output = widgets.Output()


def update_chernoff(*_):
    with chernoff_output:
        clear_output(wait=True)

        a = chernoff_a.value
        t = chernoff_t.value

        bound_t = math.exp(-t*a+t*t/2)
        optimum = math.exp(-a*a/2)
        exact_tail = 1-normal_cdf_scalar(a)

        display(Math(r"e^{-ta+t^2/2}=" + f"{bound_t:.8f}"))
        display(Math(r"\text{optimized bound}=e^{-a^2/2}=" + f"{optimum:.8f}"))
        display(Math(r"P(Z\ge a)=" + f"{exact_tail:.8f}"))


for control in (chernoff_a, chernoff_t):
    control.observe(update_chernoff, names="value")

display(widgets.VBox([
    widgets.HBox([chernoff_a,chernoff_t]),
    chernoff_output,
]))
update_chernoff()


## 7. An important limitation: MGFs need not exist

Let $X$ have the standard Cauchy density

$$
f_X(x)
=
\frac{1}{\pi(1+x^2)}.
$$

For $t>0$,

$$
M_X(t)
\ge
\frac1\pi
\int_1^\infty
\frac{e^{tx}}{1+x^2}\,dx
=
\infty.
$$

For $t<0$, the divergence occurs in the negative tail.

Therefore

$$
\boxed{
M_X(t)=\infty
\quad\text{for every }t\ne0.
}
$$


### Failure of the MGF is not failure of the distribution

The problem is the unbounded exponential weight

$$
e^{tX}.
$$

The Cauchy law is perfectly valid. We need a transform whose integrand stays bounded for every real-valued random variable.

That leads to

$$
e^{itX}.
$$


In [ ]:
cauchy_t = widgets.FloatSlider(value=0.2, min=0.05, max=1.0, step=0.05, description="t")
cauchy_cutoff = widgets.FloatSlider(value=10, min=2, max=25, step=1, description="R")
cauchy_mgf_output = widgets.Output()


def update_cauchy_mgf_growth(*_):
    with cauchy_mgf_output:
        clear_output(wait=True)

        t = cauchy_t.value
        R = cauchy_cutoff.value

        x = np.linspace(1,R,5000)
        integrand = np.exp(t*x)/(math.pi*(1+x*x))

        partial = np.trapezoid(integrand,x) if hasattr(np,"trapezoid") else np.trapz(integrand,x)

        display(Math(
            r"\frac1\pi\int_1^R\frac{e^{tx}}{1+x^2}\,dx\approx"
            + f"{partial:.6g}"
        ))

        fig, ax = plt.subplots(figsize=(8,3.2))
        ax.plot(x,integrand)
        ax.set_xlabel("x")
        ax.set_ylabel("integrand")
        ax.set_title("Positive-tail Cauchy MGF integrand")
        plt.show()


for control in (cauchy_t,cauchy_cutoff):
    control.observe(update_cauchy_mgf_growth, names="value")

display(widgets.VBox([
    widgets.HBox([cauchy_t,cauchy_cutoff]),
    cauchy_mgf_output,
]))
update_cauchy_mgf_growth()


## 8. Characteristic functions

The characteristic function of a real-valued random variable is

$$
\boxed{
\varphi_X(t)
=
\mathbb E[e^{itX}],
\qquad
t\in\mathbb R.
}
$$

Using Euler's identity,

$$
\boxed{
\varphi_X(t)
=
\mathbb E[\cos(tX)]
+
i\mathbb E[\sin(tX)].
}
$$

Sine and cosine are bounded, so the two expectations always exist.


### Every characteristic function exists

Because

$$
|e^{itX}|=1,
$$

$$
\boxed{
|\varphi_X(t)|
\le
1.
}
$$

This is why characteristic functions remain available for heavy-tailed distributions such as the Cauchy law.


### Fourier sign convention

The chapter uses

$$
\widehat\mu(t)
=
\int_{\mathbb R}
e^{itx}\,\mu(dx).
$$

Thus the characteristic function is the Fourier--Stieltjes transform of the probability law with this sign convention.


## 9. Basic properties of characteristic functions

For every random variable $X$,

$$
\boxed{
\varphi_X(0)=1,
}
$$

$$
\boxed{
\varphi_X(-t)
=
\overline{\varphi_X(t)},
}
$$

and

$$
\boxed{
\varphi_{aX+b}(t)
=
e^{ibt}\varphi_X(at).
}
$$

Every characteristic function is uniformly continuous on $\mathbb R$.


### Why uniform continuity is stronger than pointwise continuity

For any $h$,

$$
|\varphi_X(t+h)-\varphi_X(t)|
\le
\mathbb E[
|e^{ihX}-1|
].
$$

The right-hand side is independent of $t$ and tends to zero as $h\to0$ by dominated convergence.

That yields **uniform** continuity.


In [ ]:
# Symmetric two-point law.
cf_grid = np.linspace(-10,10,1200)
cf_vals = np.cos(cf_grid)

fig, ax = plt.subplots(figsize=(8,3.2))
ax.plot(cf_grid,cf_vals)
ax.set_xlabel("t")
ax.set_ylabel("phi_X(t)")
ax.set_title("Characteristic function of P(X=1)=P(X=-1)=1/2")
plt.show()

display(Math(r"\varphi_X(t)=\cos t"))


## 10. Symmetry and real-valued characteristic functions

If

$$
X\stackrel d=-X,
$$

then

$$
\boxed{
\varphi_X(t)
=
\mathbb E[\cos(tX)]
}
$$

is real valued and even.

Conversely, a real-valued characteristic function corresponds to a distribution symmetric about zero, using CF uniqueness.


In [ ]:
sym_t = np.linspace(-10,10,1000)

uniform_cf = uniform_minus1_1_cf(sym_t)

fig, ax = plt.subplots(figsize=(8,3.3))
ax.plot(sym_t,uniform_cf.real)
ax.axhline(0,linewidth=0.8)
ax.set_xlabel("t")
ax.set_ylabel("phi(t)")
ax.set_title("Real even CF of U(-1,1): sin(t)/t")
plt.show()


## 11. Moments from characteristic functions

A characteristic function always exists, but moments need not exist.

If

$$
\mathbb E[|X|^n]<\infty,
$$

then for $k=1,\ldots,n$,

$$
\boxed{
\varphi_X^{(k)}(t)
=
\mathbb E[
(iX)^k e^{itX}
].
}
$$

At the origin,

$$
\boxed{
\varphi_X^{(k)}(0)
=
i^k\mathbb E[X^k].
}
$$


### First two moments

If $\mathbb E[X^2]<\infty$, then

$$
\varphi_X'(0)
=
i\mathbb E[X],
$$

and

$$
\varphi_X''(0)
=
-\mathbb E[X^2].
$$

These formulas must not be used without checking the moment assumptions.


### Recovering moments from a normal CF

Suppose

$$
\varphi_X(t)
=
\exp\left(
2it-\frac52t^2
\right).
$$

Then uniqueness identifies

$$
X\sim N(2,5).
$$

Differentiation also gives

$$
\mathbb E[X]=2,
$$

$$
\mathbb E[X^2]=9,
$$

and therefore

$$
\operatorname{Var}(X)=5.
$$


In [ ]:
display(Math(r"\varphi_X'(0)=2i"))
display(Math(r"\varphi_X''(0)=-9"))
display(Math(r"\mathbb E[X]=2,\qquad \operatorname{Var}(X)=5"))


## 12. Characteristic functions of important distributions

The chapter derives:

$$
\operatorname{Bernoulli}(p):
\qquad
\varphi_X(t)
=
1-p+pe^{it},
$$

$$
\operatorname{Bin}(n,p):
\qquad
\varphi_X(t)
=
(1-p+pe^{it})^n,
$$

$$
\operatorname{Poisson}(\lambda):
\qquad
\varphi_X(t)
=
\exp\left(
\lambda(e^{it}-1)
\right),
$$

$$
\operatorname{Exp}(\lambda):
\qquad
\varphi_X(t)
=
\frac{\lambda}{\lambda-it},
$$

$$
N(\mu,\sigma^2):
\qquad
\varphi_X(t)
=
\exp\left(
i\mu t-\frac12\sigma^2t^2
\right),
$$

and

$$
\operatorname{Cauchy}(0,1):
\qquad
\varphi_X(t)
=
e^{-|t|}.
$$


In [ ]:
cf_family = widgets.Dropdown(
    options=[
        ("Bernoulli(0.3)","bern"),
        ("Poisson(2)","pois"),
        ("Exp(2)","exp"),
        ("N(1,1.5^2)","normal"),
        ("Standard Cauchy","cauchy"),
    ],
    value="normal",
    description="law",
)
cf_family_output = widgets.Output()


def update_cf_family(*_):
    with cf_family_output:
        clear_output(wait=True)

        t = np.linspace(-8,8,1200)

        if cf_family.value == "bern":
            vals = bernoulli_cf(t,0.3)
        elif cf_family.value == "pois":
            vals = poisson_cf(t,2)
        elif cf_family.value == "exp":
            vals = exponential_cf(t,2)
        elif cf_family.value == "normal":
            vals = normal_cf(t,1,1.5)
        else:
            vals = cauchy_cf(t)

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.plot(t,vals.real,label="Re(phi)")
        ax.plot(t,vals.imag,label="Im(phi)")
        ax.set_xlabel("t")
        ax.set_ylabel("value")
        ax.set_title("Real and imaginary parts of the characteristic function")
        ax.legend()
        plt.show()

        display(Markdown(
            f"Maximum numerical modulus on the grid: **{np.max(np.abs(vals)):.8f}**"
        ))


cf_family.observe(update_cf_family, names="value")
display(widgets.VBox([cf_family,cf_family_output]))
update_cf_family()


## 13. The normal characteristic function via an ODE

For $Z\sim N(0,1)$ with density $g$,

$$
g'(z)=-zg(z).
$$

Differentiating the CF and integrating by parts gives

$$
\varphi_Z'(t)
=
-t\varphi_Z(t).
$$

Together with

$$
\varphi_Z(0)=1,
$$

the unique solution is

$$
\boxed{
\varphi_Z(t)=e^{-t^2/2}.
}
$$


In [ ]:
ode_t = np.linspace(-4,4,1000)
phi_vals = np.exp(-ode_t**2/2)

# Numerical derivative compared with -t phi(t).
dphi = np.gradient(phi_vals,ode_t)
rhs = -ode_t*phi_vals

fig, ax = plt.subplots(figsize=(8,3.2))
ax.plot(ode_t,dphi,label="numerical phi'(t)")
ax.plot(ode_t,rhs,linestyle="--",label="-t phi(t)")
ax.set_xlabel("t")
ax.set_ylabel("value")
ax.set_title("Normal CF ODE")
ax.legend()
plt.show()

display(Markdown(
    f"Maximum grid discrepancy: **{np.max(np.abs(dphi-rhs)):.5g}**"
))


## 14. Characteristic function of the standard Cauchy distribution

For

$$
f_X(x)
=
\frac1{\pi(1+x^2)},
$$

the chapter proves, without contour integration,

$$
\boxed{
\varphi_X(t)
=
e^{-|t|}.
}
$$

The proof begins with

$$
\frac1{1+x^2}
=
\int_0^\infty
e^{-u(1+x^2)}\,du,
$$

uses Fubini, and inserts the Gaussian Fourier identity.


### The key auxiliary integral

After the Gaussian transform calculation, define

$$
I(a)
=
\int_0^\infty
u^{-1/2}
\exp\left(
-u-\frac{a^2}{4u}
\right)du.
$$

Then

$$
I(0)
=
\Gamma(1/2)
=
\sqrt\pi.
$$

For $a>0$, differentiation under the integral and the substitution

$$
u=\frac{a^2}{4v}
$$

give

$$
I'(a)=-I(a).
$$

Therefore

$$
\boxed{
I(a)
=
\sqrt\pi e^{-a}.
}
$$

Taking $a=|t|$ gives the Cauchy CF.


In [ ]:
def I_numeric(a):
    if a == 0:
        return math.sqrt(math.pi)

    # Log-spaced grid handles both the near-zero and tail regions.
    u = np.geomspace(1e-5, 40, 60000)
    integrand = u**(-0.5)*np.exp(-u-a*a/(4*u))

    if hasattr(np,"trapezoid"):
        return np.trapezoid(integrand,u)
    return np.trapz(integrand,u)


cauchy_a = widgets.FloatSlider(value=1.0, min=0, max=4, step=0.25, description="a")
cauchy_I_output = widgets.Output()


def update_cauchy_I(*_):
    with cauchy_I_output:
        clear_output(wait=True)

        a = cauchy_a.value
        numerical = I_numeric(a)
        exact = math.sqrt(math.pi)*math.exp(-a)

        display(Math(r"I(a)_{\mathrm{num}}\approx" + f"{numerical:.8f}"))
        display(Math(r"\sqrt\pi e^{-a}=" + f"{exact:.8f}"))


cauchy_a.observe(update_cauchy_I, names="value")
display(widgets.VBox([cauchy_a,cauchy_I_output]))
update_cauchy_I()


### The Cauchy contrast

The same distribution satisfies

$$
M_X(t)=\infty
\qquad
(t\ne0),
$$

but

$$
\varphi_X(t)=e^{-|t|}
$$

for every real $t$.

This is the basic reason characteristic functions are genuinely more general than MGFs.


## 15. Characteristic functions of independent sums

The real-valued expectation-factorization theorem from Chapter 11 extends componentwise to bounded complex Borel functions.

Therefore, for independent $X_1,\ldots,X_n$,

$$
\boxed{
\varphi_{X_1+\cdots+X_n}(t)
=
\prod_{j=1}^n
\varphi_{X_j}(t).
}
$$

Unlike the MGF rule, this requires no transform-existence hypothesis beyond the fact that the variables themselves exist.


### Independent normals

If

$$
X_j
\sim
N(\mu_j,\sigma_j^2)
$$

independently, then

$$
\prod_j
\exp\left(
i\mu_jt-\frac12\sigma_j^2t^2
\right)
=
\exp\left(
i\sum_j\mu_jt
-\frac12\sum_j\sigma_j^2t^2
\right).
$$

By uniqueness,

$$
\boxed{
\sum_jX_j
\sim
N\left(
\sum_j\mu_j,
\sum_j\sigma_j^2
\right).
}
$$


In [ ]:
sum_mu = [1.0,-2.0,0.5]
sum_var = [4.0,1.0,9.0]

display(Math(
    r"\sum_j\mu_j=" + f"{sum(sum_mu):.6f}"
))
display(Math(
    r"\sum_j\sigma_j^2=" + f"{sum(sum_var):.6f}"
))


## 16. Why characteristic functions determine the law

The product rule is useful for identifying distributions only because a characteristic function determines the probability law uniquely.

The chapter proves:

$$
\boxed{
\varphi_X(t)=\varphi_Y(t)
\text{ for every }t
\quad\Longrightarrow\quad
X\stackrel d=Y.
}
$$


### Gaussian smoothing

Let

$$
Z\sim N(0,1)
$$

be independent of $X$, and let $\varepsilon>0$.

Then

$$
X+\varepsilon Z
$$

always has a density, even if $X$ does not:

$$
\boxed{
f_{X,\varepsilon}(y)
=
\mathbb E[
g_\varepsilon(y-X)
].
}
$$

Adding a small independent Gaussian smooths the law.


In [ ]:
smooth_eps = widgets.FloatSlider(value=0.3, min=0.05, max=1.5, step=0.05, description="epsilon")
smooth_output = widgets.Output()


def update_smoothing(*_):
    with smooth_output:
        clear_output(wait=True)

        eps = smooth_eps.value

        # X has mass 0.4 at -1 and 0.6 at 2.
        grid = np.linspace(-4,5,1200)

        density = (
            0.4*gaussian_density(grid+1,eps)
            + 0.6*gaussian_density(grid-2,eps)
        )

        fig, ax = plt.subplots(figsize=(8,3.4))
        ax.plot(grid,density)
        ax.axvline(-1,linestyle=":")
        ax.axvline(2,linestyle=":")
        ax.set_xlabel("y")
        ax.set_ylabel("smoothed density")
        ax.set_title("Gaussian smoothing of a two-point distribution")
        plt.show()

        if hasattr(np,"trapezoid"):
            mass = np.trapezoid(density,grid)
        else:
            mass = np.trapz(density,grid)

        display(Math(r"\text{numerical total mass}\approx" + f"{mass:.8f}"))


smooth_eps.observe(update_smoothing, names="value")
display(widgets.VBox([smooth_eps,smooth_output]))
update_smoothing()


### Fourier representation of the Gaussian smoothing kernel

For $\varepsilon>0$,

$$
\boxed{
g_\varepsilon(u)
=
\frac1{2\pi}
\int_{-\infty}^{\infty}
e^{-itu}
e^{-\varepsilon^2t^2/2}\,dt.
}
$$

Combining this with Fubini gives

$$
f_{X,\varepsilon}(x)
=
\frac1{2\pi}
\int_{-\infty}^{\infty}
e^{-itx}
\varphi_X(t)
e^{-\varepsilon^2t^2/2}\,dt.
$$

Thus equal characteristic functions produce identical smoothed densities.


### Removing the smoothing

As $\varepsilon\downarrow0$,

$$
\varepsilon Z\to0
$$

in probability.

At continuity points of the cdf,

$$
P(X+\varepsilon Z\le x)
\to
F_X(x).
$$

If the smoothed distributions of $X$ and $Y$ coincide for every $\varepsilon>0$, their cdfs agree at all continuity points, and right-continuity extends the equality everywhere.


## 17. Cauchy stability under averaging

Let $X,Y$ be independent standard Cauchy variables.

Then

$$
\varphi_{X+Y}(t)
=
e^{-|t|}e^{-|t|}
=
e^{-2|t|}.
$$

If $C$ is standard Cauchy,

$$
\varphi_{2C}(t)
=
\varphi_C(2t)
=
e^{-2|t|}.
$$

By CF uniqueness,

$$
\boxed{
X+Y
\stackrel d=
2C.
}
$$

Therefore

$$
\boxed{
\frac{X+Y}{2}
\stackrel d=
C.
}
$$


### Why this matters

The average of two independent standard Cauchy variables has **exactly the same distribution** as either summand.

There is no concentration around a finite mean because the standard Cauchy distribution does not have a finite expectation.

Chapter 14 extends the same transform calculation to averages of $n$ independent standard Cauchy variables.


In [ ]:
cauchy_N = widgets.IntSlider(value=40000, min=2000, max=100000, step=2000, description="N")
cauchy_stable_output = widgets.Output()


def update_cauchy_stability(*_):
    with cauchy_stable_output:
        clear_output(wait=True)

        N = cauchy_N.value
        rng = np.random.default_rng(2026)

        X = rng.standard_cauchy(N)
        Y = rng.standard_cauchy(N)
        avg = (X+Y)/2

        q = [0.1,0.25,0.5,0.75,0.9]
        qx = np.quantile(X,q)
        qa = np.quantile(avg,q)

        display(Markdown(
            "| quantile level | one Cauchy | average of two |\n"
            "|---:|---:|---:|\n"
            + "\n".join(
                f"| {level:.2f} | {a:.4f} | {b:.4f} |"
                for level,a,b in zip(q,qx,qa)
            )
        ))

        tgrid = np.linspace(-4,4,161)
        emp = np.array([empirical_cf(avg,t) for t in tgrid])
        theory = cauchy_cf(tgrid)

        display(Markdown(
            f"Maximum empirical CF error on grid: **{np.max(np.abs(emp-theory)):.5f}**"
        ))


cauchy_N.observe(update_cauchy_stability, names="value")
display(widgets.VBox([cauchy_N,cauchy_stable_output]))
update_cauchy_stability()


## 18. Recognizing a law from its characteristic function

Suppose

$$
\varphi_X(t)
=
e^{-t^2/2}
$$

for every real $t$.

This is the standard normal characteristic function.

Uniqueness therefore gives

$$
\boxed{
X\sim N(0,1).
}
$$

No density calculation for $X$ is needed.


## 19. Uniqueness of moment generating functions

Suppose there is $\delta>0$ such that both $M_X$ and $M_Y$ are finite and

$$
M_X(t)=M_Y(t)
$$

for every

$$
|t|<\delta.
$$

Then

$$
\boxed{
X\stackrel d=Y.
}
$$

The neighborhood-of-zero assumption is essential to the theorem.


### Why the MGF theorem is not “equality at zero”

Every random variable satisfies

$$
M_X(0)=1.
$$

So equality at the single point $t=0$ contains no distributional information.

The uniqueness theorem requires equality on an open neighborhood where both transforms are finite.


## 20. Historical problem: identifying a complicated sum without repeated convolution

Suppose

$$
X_1,\ldots,X_n
\sim
\operatorname{Bernoulli}(p)
$$

independently, and let

$$
S_n=X_1+\cdots+X_n.
$$

Each summand has

$$
M_{X_j}(t)
=
1-p+pe^t.
$$

Therefore

$$
\boxed{
M_{S_n}(t)
=
(1-p+pe^t)^n.
}
$$

This is the MGF of $\operatorname{Bin}(n,p)$, so MGF uniqueness yields

$$
\boxed{
S_n\sim\operatorname{Bin}(n,p).
}
$$

The direct proof counts success patterns. The transform proof encodes each summand once and converts independence into multiplication.


### The same idea for Poisson sums

If

$$
X_j
\sim
\operatorname{Poisson}(\lambda_j)
$$

independently, then

$$
\varphi_{S_n}(t)
=
\prod_j
\exp\left(
\lambda_j(e^{it}-1)
\right)
$$

$$
=
\exp\left(
\left(
\sum_j\lambda_j
\right)
(e^{it}-1)
\right).
$$

By uniqueness,

$$
\boxed{
S_n
\sim
\operatorname{Poisson}
\left(
\sum_j\lambda_j
\right).
}
$$


## 21. Characteristic function of a sample mean

If $X_1,\ldots,X_n$ are i.i.d. with common characteristic function $\varphi$, then

$$
\overline X_n
=
\frac1n
\sum_{j=1}^nX_j
$$

has

$$
\boxed{
\varphi_{\overline X_n}(t)
=
\left[
\varphi\left(
\frac tn
\right)
\right]^n.
}
$$

Do **not** take a limit here. The limiting theory belongs to Chapter 14.


## 22. Python laboratory: empirical transforms

Given a sample $x_1,\ldots,x_N$, define

$$
\widehat M_N(t)
=
\frac1N
\sum_{j=1}^N
e^{tx_j},
$$

and

$$
\widehat\varphi_N(t)
=
\frac1N
\sum_{j=1}^N
e^{itx_j}.
$$

The second estimator is always an average of complex numbers of modulus one.


### Empirical MGF of a normal law


In [ ]:
emp_N = widgets.IntSlider(value=30000, min=1000, max=100000, step=1000, description="N")
emp_mgf_output = widgets.Output()


def update_empirical_mgf(*_):
    with emp_mgf_output:
        clear_output(wait=True)

        N = emp_N.value
        rng = np.random.default_rng(2026)

        mu = 1.0
        sigma = 1.5
        sample = rng.normal(mu,sigma,size=N)

        tgrid = np.linspace(-0.7,0.7,101)

        empirical = np.array([
            empirical_mgf(sample,t)
            for t in tgrid
        ])

        theoretical = normal_mgf(tgrid,mu,sigma)

        fig, ax = plt.subplots(figsize=(8,3.4))
        ax.plot(tgrid,theoretical,label="theoretical MGF")
        ax.plot(tgrid,empirical,linestyle="--",label="empirical MGF")
        ax.set_xlabel("t")
        ax.set_ylabel("M(t)")
        ax.set_title("Normal MGF: theory and simulation")
        ax.legend()
        plt.show()


emp_N.observe(update_empirical_mgf, names="value")
display(widgets.VBox([emp_N,emp_mgf_output]))
update_empirical_mgf()


### Empirical characteristic function of a normal law


In [ ]:
emp_cf_N = widgets.IntSlider(value=30000, min=1000, max=100000, step=1000, description="N")
emp_cf_output = widgets.Output()


def update_empirical_normal_cf(*_):
    with emp_cf_output:
        clear_output(wait=True)

        N = emp_cf_N.value
        rng = np.random.default_rng(2026)

        mu = 1.0
        sigma = 1.5
        sample = rng.normal(mu,sigma,size=N)

        tgrid = np.linspace(-6,6,121)

        empirical = np.array([
            empirical_cf(sample,t)
            for t in tgrid
        ])

        theoretical = normal_cf(tgrid,mu,sigma)

        fig, ax = plt.subplots(figsize=(8,3.5))
        ax.plot(tgrid,theoretical.real,label="theory Re(phi)")
        ax.plot(tgrid,empirical.real,linestyle="--",label="empirical Re(phi)")
        ax.plot(tgrid,theoretical.imag,label="theory Im(phi)")
        ax.plot(tgrid,empirical.imag,linestyle="--",label="empirical Im(phi)")
        ax.set_xlabel("t")
        ax.set_ylabel("value")
        ax.set_title("Normal characteristic function: theory and simulation")
        ax.legend()
        plt.show()

        display(Markdown(
            f"Maximum complex error: **{np.max(np.abs(empirical-theoretical)):.5f}**"
        ))


emp_cf_N.observe(update_empirical_normal_cf, names="value")
display(widgets.VBox([emp_cf_N,emp_cf_output]))
update_empirical_normal_cf()


### Empirical product rule for an independent sum

Let

$$
X_1\sim\operatorname{Exp}(2),
\qquad
X_2\sim\operatorname{Exp}(3)
$$

independently.

Then

$$
\varphi_{X_1+X_2}(t)
=
\frac2{2-it}
\frac3{3-it}.
$$


In [ ]:
prod_N = widgets.IntSlider(value=40000, min=2000, max=100000, step=2000, description="N")
prod_output = widgets.Output()


def update_empirical_product(*_):
    with prod_output:
        clear_output(wait=True)

        N = prod_N.value
        rng = np.random.default_rng(2026)

        x1 = rng.exponential(scale=1/2,size=N)
        x2 = rng.exponential(scale=1/3,size=N)
        s = x1+x2

        tgrid = np.linspace(-8,8,101)

        empirical = np.array([
            empirical_cf(s,t)
            for t in tgrid
        ])

        theoretical = exponential_cf(tgrid,2)*exponential_cf(tgrid,3)

        display(Markdown(
            f"Maximum empirical CF error: **{np.max(np.abs(empirical-theoretical)):.5f}**"
        ))


prod_N.observe(update_empirical_product, names="value")
display(widgets.VBox([prod_N,prod_output]))
update_empirical_product()


### Cauchy: empirical CF is stable, empirical MGF is not

For a standard Cauchy sample,

$$
\widehat\varphi_N(1)
=
\frac1N
\sum_{j=1}^N
e^{iX_j}
$$

is an average of bounded complex numbers.

By contrast,

$$
e^{tX_j}
$$

can become astronomically large after one extreme observation.

The latter sample average must **not** be treated as a consistent estimate of a nonexistent Cauchy MGF.


In [ ]:
heavy_N = widgets.IntSlider(value=20000, min=1000, max=100000, step=1000, description="N")
heavy_seed = widgets.IntSlider(value=2026, min=0, max=5000, description="seed")
heavy_output = widgets.Output()


def update_heavy_tail(*_):
    with heavy_output:
        clear_output(wait=True)

        N = heavy_N.value
        seed = heavy_seed.value
        rng = np.random.default_rng(seed)

        c = rng.standard_cauchy(size=N)

        emp_cf = empirical_cf(c,1.0)
        exact_cf = math.exp(-1)

        display(Markdown(
            f"Empirical CF at $t=1$: **{emp_cf.real:.6f} {emp_cf.imag:+.6f}i**"
        ))
        display(Math(r"e^{-1}=" + f"{exact_cf:.6f}"))
        display(Markdown(f"Largest observation: **{np.max(c):.6g}**"))
        display(Markdown(f"Smallest observation: **{np.min(c):.6g}**"))

        clipped_exp = np.exp(np.clip(0.2*c,-700,700))
        display(Markdown(
            f"Maximum clipped $e^{{0.2X}}$ in this sample: **{np.max(clipped_exp):.4e}**"
        ))


for control in (heavy_N,heavy_seed):
    control.observe(update_heavy_tail, names="value")

display(widgets.VBox([
    widgets.HBox([heavy_N,heavy_seed]),
    heavy_output,
]))
update_heavy_tail()


### Empirical CF of $U(-1,1)$

For

$$
X\sim U(-1,1),
$$

$$
\boxed{
\varphi_X(t)
=
\frac{\sin t}{t},
\qquad
t\ne0,
}
$$

with $\varphi_X(0)=1$.


In [ ]:
unif_N = widgets.IntSlider(value=30000, min=1000, max=100000, step=1000, description="N")
unif_cf_output = widgets.Output()


def update_uniform_cf(*_):
    with unif_cf_output:
        clear_output(wait=True)

        N = unif_N.value
        rng = np.random.default_rng(2026)
        sample = rng.uniform(-1,1,size=N)

        tgrid = np.linspace(-10,10,161)
        empirical = np.array([
            empirical_cf(sample,t)
            for t in tgrid
        ])

        theory = uniform_minus1_1_cf(tgrid)

        fig, ax = plt.subplots(figsize=(8,3.4))
        ax.plot(tgrid,theory,label="sin(t)/t")
        ax.plot(tgrid,empirical.real,linestyle="--",label="empirical Re(phi)")
        ax.set_xlabel("t")
        ax.set_ylabel("value")
        ax.set_title("CF of U(-1,1)")
        ax.legend()
        plt.show()

        display(Markdown(
            f"Maximum empirical complex error: **{np.max(np.abs(empirical-theory)):.5f}**"
        ))


unif_N.observe(update_uniform_cf, names="value")
display(widgets.VBox([unif_N,unif_cf_output]))
update_uniform_cf()


## 23. Transform summary

| Idea | Formula |
|---|---|
| MGF | $M_X(t)=\mathbb E[e^{tX}]$ when finite |
| MGF near zero | finite on some $(-\delta,\delta)$ |
| MGF moments | $M_X^{(n)}(0)=\mathbb E[X^n]$ |
| Independent MGF sum | $M_{\sum X_j}=\prod_jM_{X_j}$ |
| Chernoff | $P(X\ge a)\le e^{-ta}M_X(t)$ |
| Characteristic function | $\varphi_X(t)=\mathbb E[e^{itX}]$ |
| CF existence | always exists and $|\varphi_X(t)|\le1$ |
| CF conjugacy | $\varphi_X(-t)=\overline{\varphi_X(t)}$ |
| CF moments | $\varphi_X^{(k)}(0)=i^k\mathbb E[X^k]$ when the moment exists |
| Independent CF sum | $\varphi_{\sum X_j}=\prod_j\varphi_{X_j}$ |
| CF uniqueness | $\varphi_X=\varphi_Y\Rightarrow X\stackrel d=Y$ |
| Cauchy CF | $e^{-|t|}$ |
| MGF uniqueness | equality near zero plus finiteness implies equality in law |
| PGF bridge | $M_X(t)=G_X(e^t)$, $\varphi_X(t)=G_X(e^{it})$ |

The product rules are the algebraic counterparts of convolution.


## 24. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random","random"),
        ("MGF domain","domain"),
        ("MGF moments","moments"),
        ("Independent sum","sum"),
        ("Chernoff","chernoff"),
        ("CF basics","cf"),
        ("CF moments","cfmoment"),
        ("Cauchy","cauchy"),
        ("Uniqueness","unique"),
        ("PGF bridge","bridge"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "domain","moments","sum","chernoff","cf",
            "cfmoment","cauchy","unique","bridge"
        ])

    if kind == "domain":
        target = "t<3"
        prompt = "For X~Exp(3), on what range is M_X(t) finite? Enter t<3."
        hint = "The exponential MGF has denominator lambda-t."
        solution = r"t<3."

    elif kind == "moments":
        target = "0.25"
        prompt = "For X~Bernoulli(0.25), find M_X'(0)."
        hint = "The first derivative at zero is E[X]."
        solution = r"M_X'(0)=0.25."

    elif kind == "sum":
        target = "5"
        prompt = "Independent Poisson(2) and Poisson(3) variables are added. What is the new Poisson parameter?"
        hint = "Multiply transforms and combine exponents."
        solution = r"\lambda_{\mathrm{sum}}=5."

    elif kind == "chernoff":
        target = str(math.exp(-2))
        prompt = "For Z~N(0,1), use the optimized Gaussian Chernoff bound for P(Z>=2). Enter a decimal."
        hint = "Use exp(-a^2/2)."
        solution = r"P(Z\ge2)\le e^{-2}."

    elif kind == "cf":
        target = "yes"
        prompt = "Does every real-valued random variable have a characteristic function for every real t? yes/no"
        hint = "The integrand e^{itX} has modulus one."
        solution = r"\text{Yes.}"

    elif kind == "cfmoment":
        target = "-9"
        prompt = "If phi(t)=exp(2it-5t^2/2), what is phi''(0)?"
        hint = "For N(2,5), E[X^2]=5+4."
        solution = r"\varphi''(0)=-9."

    elif kind == "cauchy":
        target = "yes"
        prompt = "If X,Y are independent standard Cauchy, is (X+Y)/2 standard Cauchy? yes/no"
        hint = "Multiply e^{-|t|} twice and apply scaling."
        solution = r"\text{Yes.}"

    elif kind == "unique":
        target = "no"
        prompt = "Does equality M_X(0)=M_Y(0) imply X and Y have the same law? yes/no"
        hint = "Every MGF equals one at zero."
        solution = r"\text{No.}"

    else:
        target = "yes"
        prompt = "For a non-negative integer-valued X, does phi_X(t)=G_X(e^{it})? yes/no"
        hint = "Evaluate the complex extension of the PGF on the unit circle."
        solution = r"\text{Yes.}"

    state.clear()
    state.update(target=target,hint=hint,solution=solution)
    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].strip().lower().replace(" ","")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Check the transform domain, independence step or uniqueness hypothesis.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))

make_exercise()


## 25. AI Audit: transform arguments

Use this checklist on an AI-generated solution.

1. Is $M_X(t)$ used only where $\mathbb E[e^{tX}]$ is finite?
2. Does “MGF exists” mean existence on an open neighborhood of zero when moments or uniqueness are invoked?
3. Is equality only at $t=0$ incorrectly used to identify a law?
4. Is differentiation under the MGF justified by a neighborhood-of-zero integrability argument?
5. Is $M_X^{(n)}(0)=E[X^n]$ used only under the correct MGF hypothesis?
6. Is it recognized that MGF existence near zero implies all absolute moments?
7. Are exponential and gamma MGF domains stated correctly?
8. Is gamma parameterized by rate $\lambda$ rather than scale?
9. Is independence used exactly where the expectation of a product is factorized?
10. Is $M_{X+Y}$ incorrectly written as $M_X+M_Y$?
11. Is a sum identified from a matching MGF only after invoking MGF uniqueness?
12. Does a Chernoff argument require $M_X(t)<\infty$ at the selected $t$?
13. Is a Chernoff bound being confused with an exact tail probability?
14. Is the Cauchy MGF correctly recognized as divergent for every $t\ne0$?
15. Is every characteristic function recognized as existing for every real $t$?
16. Is the bound $|\varphi_X(t)|\le1$ justified by $|e^{itX}|=1$?
17. Is $\varphi_X(-t)=\overline{\varphi_X(t)}$ used correctly?
18. Is uniform continuity distinguished from mere pointwise continuity?
19. Are CF derivatives used only when the corresponding absolute moments exist?
20. Is the normal CF derived with the correct sign $-\sigma^2t^2/2$?
21. Is the standard Cauchy CF written as $e^{-|t|}$, not $e^{-t}$?
22. Is complex expectation factorization justified under independence?
23. Is a distribution identified from its CF only after using uniqueness?
24. Is Gaussian smoothing understood as adding independent normal noise?
25. Is MGF uniqueness stated with equality on a neighborhood of zero and finiteness there?
26. Is the Cauchy stability calculation correctly giving $(X+Y)/2\stackrel d=C$?
27. Is a nonexistent Cauchy MGF being “estimated” from a finite sample?
28. Are empirical transforms clearly distinguished from exact transform identities?
29. Is a sample-mean CF formula stopped before taking a limit that belongs to Chapter 14?
30. Is the PGF--MGF--CF bridge used only in the appropriate integer-valued setting?

### Claims to audit

- “Every random variable has an MGF on all of $\mathbb R$.”
- “For independent $X,Y$, $M_{X+Y}=M_X+M_Y$.”
- “A Cauchy distribution has no characteristic function because its mean does not exist.”
- “If two MGFs agree at $t=0$, the distributions are equal.”
- “The standard Cauchy CF is $e^{-t}$.”
- “The empirical MGF of a heavy-tailed Cauchy sample proves that the Cauchy MGF exists.”

All six claims are false.


### Suggested AI-guided activities

- “Give me one discrete and one continuous law and make me determine the MGF domain before differentiating anything.”
- “Guide me through the exact point where independence is used in the transform of a sum.”
- “Start with the standard Cauchy law and make me contrast $e^{tX}$ with $e^{itX}$ before computing any transform.”
- “Make me verify $|\varphi(t)|\le1$, conjugacy, and uniform continuity from the definition.”
- “Guide me through Gaussian smoothing as the key step in CF uniqueness.”
- “Give me two independent Cauchy variables and require a full transform/uniqueness proof of stability under averaging.”
- “Simulate empirical MGFs and CFs and force me to state what is an exact theorem and what is numerical evidence.”


## 26. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. M_X(0) equals:",
        ["Choose...","0","1","E[X]"],
        "1",
        r"M_X(0)=\mathbb E[1]=1.",
    ),
    (
        "2. Useful MGF existence for moment theory means:",
        ["Choose...","finite only at 0","finite on a neighborhood of 0"],
        "finite on a neighborhood of 0",
        r"\exists\delta>0:\ M_X(t)<\infty\text{ for }|t|<\delta.",
    ),
    (
        "3. If the MGF exists near zero, M_X'(0) equals:",
        ["Choose...","E[X]","Var(X)","1"],
        "E[X]",
        r"M_X'(0)=\mathbb E[X].",
    ),
    (
        "4. Independent sums satisfy:",
        ["Choose...","M_{X+Y}=M_XM_Y","M_{X+Y}=M_X+M_Y"],
        "M_{X+Y}=M_XM_Y",
        r"\text{Independence factorizes the expectation.}",
    ),
    (
        "5. Every random variable has a characteristic function:",
        ["Choose...","true","false"],
        "true",
        r"|e^{itX}|=1.",
    ),
    (
        "6. Every characteristic function satisfies:",
        ["Choose...","|phi(t)|<=1","phi(t)>=0","phi(t) is real"],
        "|phi(t)|<=1",
        r"|\varphi_X(t)|\le1.",
    ),
    (
        "7. If E[X^2]<infinity, then phi''(0) equals:",
        ["Choose...","-E[X^2]","E[X^2]","iE[X^2]"],
        "-E[X^2]",
        r"\varphi_X''(0)=-\mathbb E[X^2].",
    ),
    (
        "8. The standard normal CF is:",
        ["Choose...","exp(-t^2/2)","exp(t^2/2)","exp(-|t|)"],
        "exp(-t^2/2)",
        r"\varphi_Z(t)=e^{-t^2/2}.",
    ),
    (
        "9. The standard Cauchy CF is:",
        ["Choose...","exp(-|t|)","exp(-t^2/2)","undefined"],
        "exp(-|t|)",
        r"\varphi_C(t)=e^{-|t|}.",
    ),
    (
        "10. Equality of characteristic functions for all t implies equality in law:",
        ["Choose...","true","false"],
        "true",
        r"\varphi_X=\varphi_Y\Longrightarrow X\stackrel d=Y.",
    ),
    (
        "11. Equality of MGFs only at t=0 implies equality in law:",
        ["Choose...","true","false"],
        "false",
        r"\text{Every MGF equals one at zero.}",
    ),
    (
        "12. If X,Y are iid standard Cauchy, then (X+Y)/2 is standard Cauchy:",
        ["Choose...","true","false"],
        "true",
        r"\varphi_{(X+Y)/2}(t)=e^{-|t|}.",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    d = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="500px"),
    )
    quiz_widgets.append(d)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:690px'>{prompt}</div>"),
        d,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_,_,correct,_) in zip(quiz_widgets,quiz_data)
        )

        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i,(widget,(_,_,correct,explanation)) in enumerate(
            zip(quiz_widgets,quiz_data),1
        ):
            mark = "✓" if widget.value == correct else "✗"

            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))
            display(Math(explanation))


grade_button.on_click(grade_quiz)

display(widgets.VBox(
    quiz_rows+[grade_button,quiz_output]
))


## 27. Automatic mathematical verification

The final code cell checks representative formulas from the chapter.


In [ ]:
# PGF/MGF/CF Bernoulli bridge.
p = 0.3
t = 0.7

G_et = 1-p+p*math.exp(t)
G_eit = 1-p+p*cmath.exp(1j*t)

assert abs(G_et-bernoulli_mgf(t,p)) < 1e-14
assert abs(G_eit-bernoulli_cf(t,p)) < 1e-14

# MGF moments.
assert abs(p-p) < 1e-14
assert abs(p-p*p-p*(1-p)) < 1e-14

# Poisson product rule.
l1,l2 = 2.0,3.0

for t in [-0.5,0,0.4,1.0]:
    lhs = poisson_mgf(t,l1)*poisson_mgf(t,l2)
    rhs = poisson_mgf(t,l1+l2)
    assert np.isclose(lhs,rhs,rtol=1e-12,atol=1e-12)

# Gamma moments.
alpha,lam = 3.0,2.0
assert abs(alpha/lam-1.5) < 1e-12
assert abs(alpha/(lam*lam)-0.75) < 1e-12

# Normal affine rule.
a,b = -2.0,5.0
mu,sigma = 2.0,3.0
t = 0.2

lhs = normal_mgf(t,a*mu+b,abs(a)*sigma)
rhs = math.exp(b*t)*normal_mgf(a*t,mu,sigma)

assert abs(lhs-rhs) < 1e-12

# CF bound and conjugacy for several families.
tgrid = np.linspace(-8,8,2001)

families = [
    bernoulli_cf(tgrid,0.3),
    poisson_cf(tgrid,2),
    exponential_cf(tgrid,2),
    normal_cf(tgrid,1,1.5),
    cauchy_cf(tgrid),
]

for vals in families:
    assert np.max(np.abs(vals)) <= 1+1e-12

# Normal CF conjugacy.
positive = normal_cf(tgrid,1,1.5)
negative = normal_cf(-tgrid,1,1.5)

assert np.allclose(negative,np.conj(positive))

# Uniform CF is real and even.
uvals = uniform_minus1_1_cf(tgrid)

assert np.max(np.abs(uvals.imag)) < 1e-14
assert np.allclose(uvals,uniform_minus1_1_cf(-tgrid))

# Normal CF derivatives at zero from parameters.
mu = 2
var = 5
EX2 = var+mu*mu

assert EX2 == 9

# Cauchy CF.
assert abs(float(cauchy_cf(np.array([0]))[0])-1) < 1e-15
assert abs(float(cauchy_cf(np.array([1]))[0])-math.exp(-1)) < 1e-15
assert np.allclose(cauchy_cf(tgrid),cauchy_cf(-tgrid))

# Cauchy independent-sum scaling.
for t in [-3,-1,0,0.7,2]:
    sum_cf = math.exp(-abs(t))*math.exp(-abs(t))
    scaled_cf = math.exp(-abs(2*t))
    assert abs(sum_cf-scaled_cf) < 1e-15

# Gaussian smoothing of a discrete law integrates approximately to one.
grid = np.linspace(-8,8,100000)
eps = 0.4

density = (
    0.4*gaussian_density(grid+1,eps)
    + 0.6*gaussian_density(grid-2,eps)
)

if hasattr(np,"trapezoid"):
    mass = np.trapezoid(density,grid)
else:
    mass = np.trapz(density,grid)

assert abs(mass-1) < 1e-8

# Gaussian Chernoff is indeed an upper bound.
for a in [0.5,1,2,3]:
    exact = 1-normal_cdf_scalar(a)
    bound = math.exp(-a*a/2)
    assert exact <= bound+1e-15

# Sample-mean CF identity algebraically for a normal law.
mu,sigma = 1.0,2.0
n = 7

for t in [-2,-0.4,0,0.8,3]:
    lhs = normal_cf(t,mu,sigma/math.sqrt(n))
    rhs = normal_cf(t/n,mu,sigma)**n

    # Mean of n iid N(mu,sigma^2) is N(mu,sigma^2/n).
    assert abs(lhs-rhs) < 1e-12

show_result(
    "All Chapter 13 automatic checks passed",
    r"M_X^{(n)}(0)=\mathbb E[X^n]\quad\text{when the MGF exists near zero}",
    r"M_{\sum X_j}(t)=\prod_jM_{X_j}(t)",
    r"|\varphi_X(t)|\le1",
    r"\varphi_X(-t)=\overline{\varphi_X(t)}",
    r"\varphi_{\mathrm{Cauchy}}(t)=e^{-|t|}",
    r"\varphi_X=\varphi_Y\Longrightarrow X\stackrel d=Y",
    note=(
        "Bridge formulas, transform products, CF structure, Cauchy stability, "
        "Gaussian smoothing and Chernoff checks all passed."
    ),
)


## 28. Chapter map

| Chapter concept | Computational representation |
|---|---|
| PGF/MGF/CF bridge | Bernoulli transform identity |
| MGF | direct definition and domain |
| MGF near zero | explicit neighborhood condition |
| affine MGF | normal transformation check |
| MGF derivatives | Bernoulli moments |
| absolute moments | domination consequence |
| standard MGFs | interactive formulas and domains |
| independent sums | Poisson, gamma and normal transform products |
| Chernoff | optimized Gaussian bound |
| MGF limitation | Cauchy divergence visualization |
| characteristic function | complex expectation |
| CF existence | modulus-one integrand |
| CF conjugacy | real/imaginary plots |
| uniform continuity | DCT-based structural argument |
| symmetry | real even CF |
| CF derivatives | normal moment recovery |
| standard CFs | Bernoulli, Poisson, exponential, normal, Cauchy |
| normal CF | ODE visualization |
| standard Cauchy CF | numerical $I(a)$ identity |
| complex factorization | product rule for independent sums |
| CF uniqueness | Gaussian smoothing |
| Cauchy stability | average of independent Cauchy variables |
| MGF uniqueness | neighborhood-of-zero hypothesis |
| historical transform problem | Bernoulli and Poisson sums |
| sample mean transform | $[\varphi(t/n)]^n$ without taking a limit |
| empirical MGF | normal simulation |
| empirical CF | normal and uniform simulations |
| heavy-tail contrast | stable empirical Cauchy CF versus unstable exponentials |
| AI Audit | transform-domain, independence and uniqueness checks |

The chapter's central computational principle is:

$$
\boxed{
\text{convolution in probability space}
\quad\longrightarrow\quad
\text{multiplication in transform space}.
}
$$

Characteristic functions make this principle universal because they exist for every probability law.
